In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import requests

#### What are the libraries we use to get data from API in python

Requests,urllib3,HTTPX

#### What is an api

a set of rules and protocols that allow different software applications to communicate with each other

#### What are api endpoints

a specific URL or digital location where an Application Programming Interface (API) receives requests and sends responses

#### What is rest api

a set of rules and design principles that allows different software applications to communicate with each other over the internet

#### What it is called when we get data from an API

API call

#### Can we use agents to read data from an API

yes

In [11]:
import pandas as pd
from urllib.request import urlopen
import json

In [14]:
import requests
import pandas as pd

# ---------------------------------------------------------
# STEP 1: Get the Session Key for the Monaco Grand Prix
# ---------------------------------------------------------
sessions_url = "https://api.openf1.org/v1/sessions"

# Filter for Monaco, a specific year, and the Race session
session_params = {
    "country_name": "Monaco",
    "year": 2024, # Adjust year as needed
    "session_name": "Race" # Change to "Qualifying" for your prediction model
}

print("Fetching session key...")
session_response = requests.get(sessions_url, params=session_params)
session_data = session_response.json()

# Extract the session key from the first matching result
if session_data:
    monaco_session_key = session_data[0]['session_key']
    print(f"Found Monaco Session Key: {monaco_session_key}")
else:
    print("Session not found.")
    exit()

# ---------------------------------------------------------
# STEP 2: Get the Final Positions using Pandas
# ---------------------------------------------------------
# OpenF1 records every single position change. We will pull 
# all position data for the session and extract the final one.
positions_url = "https://api.openf1.org/v1/position"
position_params = {
    "session_key": monaco_session_key
}

print("Fetching position data...")
position_response = requests.get(positions_url, params=position_params)
position_data = position_response.json()

# Load into a DataFrame
df_positions = pd.DataFrame(position_data)

# Convert the date column to actual datetime objects for accurate sorting
df_positions['date'] = pd.to_datetime(df_positions['date'])

# Sort by driver and then by time to get the chronological order of positions
df_positions = df_positions.sort_values(by=['driver_number', 'date'])

# Keep only the LAST recorded position for each driver (which is the final race result)
final_results = df_positions.drop_duplicates(subset=['driver_number'], keep='last')

# Sort the final dataframe by position to show 1st to 20th
final_results = final_results.sort_values(by='position').reset_index(drop=True)

# Display the top 5
print("\nFinal Monaco GP Top 5:")
print(final_results[['position', 'driver_number', 'date']].head())

Fetching session key...
Found Monaco Session Key: 9523
Fetching position data...

Final Monaco GP Top 5:
   position  driver_number                             date
0         1             16 2024-05-26 12:08:08.494000+00:00
1         2             81 2024-05-26 12:08:08.494000+00:00
2         3             55 2024-05-26 13:44:14.776000+00:00
3         4              4 2024-05-26 13:44:18.547000+00:00
4         5             63 2024-05-26 13:44:18.547000+00:00


In [24]:
drivers_url = f"https://api.openf1.org/v1/drivers?session_key={monaco_session_key}"
drivers_data = requests.get(drivers_url).json()
 # Print the first 10 entries to check the structure
print(df_drivers.head(10))

# Convert to DataFrame
df_drivers = pd.DataFrame(drivers_data)

# Merge the final results with driver details (Name, Team, Acronym)
complete_results = pd.merge(
    final_results, 
    df_drivers[['driver_number', 'full_name', 'team_name']], 
    on='driver_number', 
    how='left'
)

print(complete_results[['position', 'full_name', 'team_name']])

   meeting_key  session_key  driver_number broadcast_name         full_name  \
0         1236         9523              1   M VERSTAPPEN    Max VERSTAPPEN   
1         1236         9523              2     L SARGEANT    Logan SARGEANT   
2         1236         9523              3    D RICCIARDO  Daniel RICCIARDO   
3         1236         9523              4       L NORRIS      Lando NORRIS   
4         1236         9523             10        P GASLY      Pierre GASLY   
5         1236         9523             11        S PEREZ      Sergio PEREZ   
6         1236         9523             14       F ALONSO   Fernando ALONSO   
7         1236         9523             16      C LECLERC   Charles LECLERC   
8         1236         9523             18       L STROLL      Lance STROLL   
9         1236         9523             20    K MAGNUSSEN   Kevin MAGNUSSEN   

  name_acronym        team_name team_colour first_name   last_name  \
0          VER  Red Bull Racing      3671C6        Max  Vers

In [17]:
import requests
import pandas as pd
import time

# ---------------------------------------------------------
# STEP 1: Get all Race sessions for 2023
# ---------------------------------------------------------
print("Fetching 2023 session keys...")
sessions_url = "https://api.openf1.org/v1/sessions"
# Tip: Change "Race" to "Qualifying" to pull the quali results for your prediction dataset
session_params = {
    "year": 2023,
    "session_name": "Race" 
}

session_response = requests.get(sessions_url, params=session_params)
sessions_data = session_response.json()

# Create a list of dictionaries containing the session_key and the country (for labeling)
race_sessions = [{'session_key': s['session_key'], 'country': s['country_name']} for s in sessions_data]
print(f"Found {len(race_sessions)} races for 2023.\n")

all_2023_results = []

# ---------------------------------------------------------
# STEP 2: Loop through each race to get the final positions
# ---------------------------------------------------------
for race in race_sessions:
    session_key = race['session_key']
    country = race['country']
    print(f"Processing {country} Grand Prix...")
    
    # Fetch position data
    positions_url = "https://api.openf1.org/v1/position"
    pos_params = {"session_key": session_key}
    
    pos_response = requests.get(positions_url, params=pos_params)
    
    # Check if the endpoint successfully returned data
    if pos_response.status_code == 200 and pos_response.json():
        df_positions = pd.DataFrame(pos_response.json())
        
        # --- THE FIX ---
        # Parse the datetime, telling Pandas to expect variations of ISO8601 formats
        df_positions['date'] = pd.to_datetime(df_positions['date'], format='ISO8601')
        
        # Sort chronologically by driver and time
        df_positions = df_positions.sort_values(by=['driver_number', 'date'])
        
        # Keep the last position recorded for each driver (the final result)
        final_results = df_positions.drop_duplicates(subset=['driver_number'], keep='last').copy()
        
        # Add the country name so we know which race this is in our final dataset
        final_results['race'] = country
        
        all_2023_results.append(final_results)
        
    # Brief pause to avoid hammering the API and getting rate-limited
    time.sleep(0.5)

# ---------------------------------------------------------
# STEP 3: Combine everything into one master DataFrame
# ---------------------------------------------------------
if all_2023_results:
    master_df = pd.concat(all_2023_results, ignore_index=True)
    
    # Clean up the dataset to keep only the columns we care about
    master_df = master_df[['race', 'position', 'driver_number']]
    
    # Sort the master list by Race, then by Position (1st to 20th)
    master_df = master_df.sort_values(by=['race', 'position']).reset_index(drop=True)
    
    print("\nExtraction Complete! Here is a sample of the master dataset:")
    print(master_df.head(15))
    
    # Uncomment the line below to export to CSV
    # master_df.to_csv("f1_2023_race_results.csv", index=False)
    # print("Saved to f1_2023_race_results.csv")
else:
    print("\nNo data extracted.")

Fetching 2023 session keys...
Found 23 races for 2023.

Processing Bahrain Grand Prix...
Processing Saudi Arabia Grand Prix...
Processing Australia Grand Prix...
Processing Azerbaijan Grand Prix...
Processing United States Grand Prix...
Processing Italy Grand Prix...
Processing Monaco Grand Prix...
Processing Spain Grand Prix...
Processing Canada Grand Prix...
Processing Austria Grand Prix...
Processing United Kingdom Grand Prix...
Processing Hungary Grand Prix...
Processing Belgium Grand Prix...
Processing Netherlands Grand Prix...
Processing Italy Grand Prix...
Processing Singapore Grand Prix...
Processing Japan Grand Prix...
Processing Qatar Grand Prix...
Processing United States Grand Prix...
Processing Mexico Grand Prix...
Processing Brazil Grand Prix...
Processing United States Grand Prix...
Processing United Arab Emirates Grand Prix...

Extraction Complete! Here is a sample of the master dataset:
         race  position  driver_number
0   Australia         1              1
1   A

In [18]:
import requests
import pandas as pd
import time

def fetch_f1_data():
    # ---------------------------------------------------------
    # STEP 1: Get all Race sessions for 2023
    # ---------------------------------------------------------
    print("Fetching 2023 session keys...")
    sessions_url = "https://api.openf1.org/v1/sessions"
    
    # Change "Race" to "Qualifying" if you want qualifying results
    session_params = {"year": 2023, "session_name": "Race"}
    session_response = requests.get(sessions_url, params=session_params)
    
    if session_response.status_code != 200:
        print("Failed to fetch sessions.")
        return
        
    sessions_data = session_response.json()
    race_sessions = [{'session_key': s['session_key'], 'country': s['country_name']} for s in sessions_data]
    print(f"Found {len(race_sessions)} races for 2023.\n")

    all_2023_results = []

    # ---------------------------------------------------------
    # STEP 2: Loop through each race to build the dataset
    # ---------------------------------------------------------
    for race in race_sessions:
        session_key = race['session_key']
        country = race['country']
        print(f"Processing {country} Grand Prix...")
        
        # --- A. Fetch Final Positions ---
        pos_url = f"https://api.openf1.org/v1/position?session_key={session_key}"
        pos_response = requests.get(pos_url)
        
        if pos_response.status_code == 200 and pos_response.json():
            df_positions = pd.DataFrame(pos_response.json())
            
            # Fix datetime parsing error by allowing mixed ISO8601 formats
            df_positions['date'] = pd.to_datetime(df_positions['date'], format='ISO8601')
            df_positions = df_positions.sort_values(by=['driver_number', 'date'])
            
            # Keep the last recorded position for the final result
            final_results = df_positions.drop_duplicates(subset=['driver_number'], keep='last').copy()
            
            # --- B. Fetch Driver & Team Info ---
            drivers_url = f"https://api.openf1.org/v1/drivers?session_key={session_key}"
            drivers_response = requests.get(drivers_url)
            
            if drivers_response.status_code == 200 and drivers_response.json():
                df_drivers = pd.DataFrame(drivers_response.json())
                df_drivers = df_drivers[['driver_number', 'full_name', 'team_name', 'name_acronym']]
                final_results = pd.merge(final_results, df_drivers, on='driver_number', how='left')

            # --- C. Fetch Lap Data to Calculate Pace Features ---
            laps_url = f"https://api.openf1.org/v1/laps?session_key={session_key}"
            laps_response = requests.get(laps_url)
            
            if laps_response.status_code == 200 and laps_response.json():
                df_laps = pd.DataFrame(laps_response.json())
                
                if 'lap_duration' in df_laps.columns:
                    df_laps['lap_duration'] = pd.to_numeric(df_laps['lap_duration'], errors='coerce')
                    
                    # Group by driver to calculate pace stats
                    pace_stats = df_laps.groupby('driver_number')['lap_duration'].agg(
                        fastest_lap='min',
                        avg_lap_time='mean',
                        laps_completed='count'
                    ).reset_index()
                    
                    final_results = pd.merge(final_results, pace_stats, on='driver_number', how='left')

            # Add race identifier
            final_results['race'] = country
            all_2023_results.append(final_results)
            
        # Pause to respect API rate limits
        time.sleep(1.0)

    # ---------------------------------------------------------
    # STEP 3: Combine and Clean Master Dataset
    # ---------------------------------------------------------
    if all_2023_results:
        master_df = pd.concat(all_2023_results, ignore_index=True)
        
        # Reorder columns logically
        columns_to_keep = [
            'race', 'position', 'driver_number', 'name_acronym', 'team_name', 
            'laps_completed', 'fastest_lap', 'avg_lap_time'
        ]
        
        # Ensure we only keep columns that actually exist
        final_cols = [col for col in columns_to_keep if col in master_df.columns]
        master_df = master_df[final_cols]
        
        # Sort by Race, then by finishing Position
        master_df = master_df.sort_values(by=['race', 'position']).reset_index(drop=True)
        
        print("\nExtraction Complete! Here is a sample of the enriched dataset:")
        print(master_df.head(15))
        
        # Export to CSV for your machine learning pipeline
        master_df.to_csv("f1_2023_dataset.csv", index=False)
        print("\n✅ Data successfully saved to 'f1_2023_dataset.csv'")
    else:
        print("\nNo data extracted. Check your connection or API status.")

# Run the script
if __name__ == "__main__":
    fetch_f1_data()

Fetching 2023 session keys...
Found 23 races for 2023.

Processing Bahrain Grand Prix...
Processing Saudi Arabia Grand Prix...
Processing Australia Grand Prix...
Processing Azerbaijan Grand Prix...
Processing United States Grand Prix...
Processing Italy Grand Prix...
Processing Monaco Grand Prix...
Processing Spain Grand Prix...
Processing Canada Grand Prix...
Processing Austria Grand Prix...
Processing United Kingdom Grand Prix...
Processing Hungary Grand Prix...
Processing Belgium Grand Prix...
Processing Netherlands Grand Prix...
Processing Italy Grand Prix...
Processing Singapore Grand Prix...
Processing Japan Grand Prix...
Processing Qatar Grand Prix...
Processing United States Grand Prix...
Processing Mexico Grand Prix...
Processing Brazil Grand Prix...
Processing United States Grand Prix...
Processing United Arab Emirates Grand Prix...

Extraction Complete! Here is a sample of the enriched dataset:
         race  position  driver_number name_acronym        team_name  \
0   Austr

In [21]:
from urllib.request import urlopen
import json

response = urlopen('https://api.openf1.org/v1/laps?session_key=9839&driver_number=44&lap_number<=3')
data = json.loads(response.read().decode('utf-8'))
print(data)

# If you want, you can import the results in a DataFrame (you need to install the `pandas` package first)
# import pandas as pd
df = pd.DataFrame(data)
print(df.head())

[{'meeting_key': 1276, 'session_key': 9839, 'driver_number': 44, 'lap_number': 1, 'date_start': '2025-12-07T13:03:27.584000+00:00', 'duration_sector_1': 24.475, 'duration_sector_2': 39.82, 'duration_sector_3': 35.449, 'i1_speed': 279, 'i2_speed': 313, 'is_pit_out_lap': False, 'lap_duration': 99.744, 'segments_sector_1': [2049, 2049, 2049, 2049, 2049], 'segments_sector_2': [2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049], 'segments_sector_3': [2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049], 'st_speed': 315}, {'meeting_key': 1276, 'session_key': 9839, 'driver_number': 44, 'lap_number': 2, 'date_start': '2025-12-07T13:05:07.327000+00:00', 'duration_sector_1': 18.501, 'duration_sector_2': 38.405, 'duration_sector_3': 33.947, 'i1_speed': 287, 'i2_speed': 320, 'is_pit_out_lap': False, 'lap_duration': 90.853, 'segments_sector_1': [None, 2049, 2049, 2049, 2049], 'segments_sector_2': [2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049, 2049], 'segments_sector_3': [2049, 2049, 204